# 13a — Rating Mirror dual-channel RBM training (5k)

Masked CD-1 training on notebook **12a** tensors:

- **RBM_A1:** user-rating one-hot (`channel1` / A1, visible = `n_movies × K`)
- **RBM_A2:** mirror complement one-hot (`channelA2`, visible = `n_movies × K`)

Same training logic as notebook **13b**, except channel 2 is Rating Mirror (same visible size as channel 1).  
**No** `deltaW` logging.

| Part | Section |
|------|--------|
| Part 0 | Setup |
| Part 1 | Load / flatten / masks |
| Part 2 | Masked CD-1 training |
| Part 3 | Save `_5k` artifacts |
| Part 3.5 | Mixed-sign occupancy |
| Part 4 | Verification |

**Prerequisites:** notebooks 09b, 12a.

## Part 0 — Setup

In [1]:
from pathlib import Path
import time

import numpy as np

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

proc = root / "data" / "processed"
assert proc.exists(), f"Missing {proc}"

N_HIDDEN = 128
LR = 0.01
BATCH_SIZE = 500
EPOCHS = 10
INIT_SEED = 42
K = 10
X_DTYPE = np.float32
N_USERS_MIRROR = 2000

path_a1 = proc / "channel1_softmax.npy"
path_a2 = proc / "channelA2_mirror.npy"
path_mask = proc / "mask.npy"
for pth in (path_a1, path_a2, path_mask):
    assert pth.exists(), f"Missing {pth} — run notebooks 09b / 12a first."

ch1_meta = np.load(path_a1, mmap_mode="r")
ch2_meta = np.load(path_a2, mmap_mode="r")
mask_meta = np.load(path_mask, mmap_mode="r")
n_users, n_movies, k_b1 = ch1_meta.shape
assert k_b1 == K
assert ch2_meta.shape == (n_users, n_movies, K)
assert mask_meta.shape == (n_users, n_movies)
assert N_USERS_MIRROR <= n_users, f"N_USERS_MIRROR={N_USERS_MIRROR} > n_users={n_users}"

n_vis = n_movies * K
bytes_x = N_USERS_MIRROR * n_vis * np.dtype(X_DTYPE).itemsize

print(f"Project root: {root}")
print(f"Full cohort: n_users={n_users:,}, n_movies={n_movies:,}, K={K}")
print(f"Training subset: N_USERS_MIRROR={N_USERS_MIRROR:,} (first {N_USERS_MIRROR} rows)")
print(f"Visible size (both channels): {n_vis:,}")
print(f"X1 / X2 each (~{X_DTYPE.__name__}, subset): {bytes_x / 1e9:.2f} GB")
print(f"Hyperparams: N_HIDDEN={N_HIDDEN}, LR={LR}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, SEED={INIT_SEED}")
print("No deltaW logging.")
print("Proceed to Part 1.")

del ch1_meta, ch2_meta, mask_meta


Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
Full cohort: n_users=5,000, n_movies=13,129, K=10
Training subset: N_USERS_MIRROR=2,000 (first 2000 rows)
Visible size (both channels): 131,290
X1 / X2 each (~float32, subset): 1.05 GB
Hyperparams: N_HIDDEN=128, LR=0.01, BATCH_SIZE=500, EPOCHS=10, SEED=42
No deltaW logging.
Proceed to Part 1.


## Part 1 — Load data / flatten / masks

In [2]:
print("Loading first N_USERS_MIRROR rows via memmap (avoids ~5GB full-tensor RAM spike) …")
channelA1_mm = np.load(path_a1, mmap_mode="r")
channelA2_mm = np.load(path_a2, mmap_mode="r")
mask_mm = np.load(path_mask, mmap_mode="r")

n_users_full, n_movies, k_dim = channelA1_mm.shape
assert channelA2_mm.shape == (n_users_full, n_movies, K)
assert mask_mm.shape == (n_users_full, n_movies)
assert N_USERS_MIRROR <= n_users_full

# Materialize ONLY the training subset (~1GB each, not full 2.6GB×2)
X1 = np.asarray(channelA1_mm[:N_USERS_MIRROR], dtype=X_DTYPE).reshape(N_USERS_MIRROR, -1)
X2 = np.asarray(channelA2_mm[:N_USERS_MIRROR], dtype=X_DTYPE).reshape(N_USERS_MIRROR, -1)
mask = np.asarray(mask_mm[:N_USERS_MIRROR])
del channelA1_mm, channelA2_mm, mask_mm

M = np.repeat(mask.astype(X_DTYPE, copy=False), K, axis=1)

print(f"X1 {X1.shape} {X1.dtype}")
print(f"X2 {X2.shape} {X2.dtype}")
print(f"M  {M.shape} {M.dtype}")

assert X1.shape == X2.shape, f"X1 {X1.shape} != X2 {X2.shape}"
assert M.shape == X1.shape
assert X1.shape[0] == N_USERS_MIRROR

nz_x1 = int((X1 > 0).sum())
nz_x2 = int((X2 > 0).sum())
nz_obs = int(mask.sum())
assert nz_x1 == nz_x2, f"nonzero mass X1={nz_x1}, X2={nz_x2}"
assert nz_x1 == nz_obs, f"X nonzero {nz_x1} != mask sum {nz_obs}"
# Do NOT globally sum float32 M — exceeds float32 exact-integer range (2^24).
assert np.allclose(
    M.sum(axis=1),
    K * mask.astype(np.float64).sum(axis=1),
), "per-user M sum must equal K × mask"
print(f"✓ X1/X2 shape match; one-hot nnz={nz_x1:,}; M = K×mask per user (skip global float32 sum)")
nz_movies = mask.sum(axis=1)
print(
    f"  train movies/user: min={int(nz_movies.min())}, "
    f"median={float(np.median(nz_movies)):.0f}, max={int(nz_movies.max())}"
)


Loading first N_USERS_MIRROR rows via memmap (avoids ~5GB full-tensor RAM spike) …
X1 (2000, 131290) float32
X2 (2000, 131290) float32
M  (2000, 131290) float32
✓ X1/X2 shape match; one-hot nnz=2,298,534; M = K×mask per user (skip global float32 sum)
  train movies/user: min=773, median=995, max=6032


## Part 2 — Masked CD-1 training

After negative-phase `v_recon_prob` / `v_recon`, multiply by the batch mask so test/unrated visibles contribute no gradient.  
Same `train_rbm` as 13b (sigmoid clip, masked recon MSE at epochs 1/25/50, per-epoch timing) — **without** deltaW logging.

In [3]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))


def reconstruction_mse(X, M, W, b_h, chunk=250):
    """Masked MSE in row-chunks — avoids a full (n_users, n_visible) recon allocation."""
    n = X.shape[0]
    num = 0.0
    den = 0.0
    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n)
        x = X[i0:i1]
        m = M[i0:i1]
        h_prob = sigmoid(x @ W + b_h)
        v_prob = sigmoid(h_prob @ W.T)
        diff2 = (x - v_prob) ** 2
        num += float((diff2 * m).sum())
        den += float(m.sum())
    return num / den if den > 0 else float("nan")


def train_rbm(X, M, n_hidden, seed, channel_name):
    """Masked CD-1; return W, b_h, mse_log, elapsed_sec. (No deltaW logging.)"""
    rng = np.random.default_rng(seed)
    n_samples, n_visible = X.shape
    assert M.shape == X.shape, f"Mask shape {M.shape} != X shape {X.shape}"

    W = rng.normal(0.0, 0.01, size=(n_visible, n_hidden)).astype(np.float64)
    b_h = np.zeros(n_hidden, dtype=np.float64)

    mse_log = {}
    t0_all = time.perf_counter()

    print(f"\n=== Training {channel_name} ===", flush=True)
    print(f"  samples={n_samples}, visible={n_visible}, hidden={n_hidden}", flush=True)
    print(f"  lr={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}, init_seed={seed}", flush=True)

    for epoch in range(EPOCHS):
        t0 = time.perf_counter()
        print(f"  … epoch {epoch + 1}/{EPOCHS} training", flush=True)
        epoch_delta = np.zeros_like(W)
        order = rng.permutation(n_samples)

        for start in range(0, n_samples, BATCH_SIZE):
            idx = order[start : start + BATCH_SIZE]
            v_data = X[idx]
            m_batch = M[idx]
            bs = v_data.shape[0]

            # Positive phase (v_data already zero on test/unrated)
            h_prob = sigmoid(v_data @ W + b_h)
            h_data = (rng.random(h_prob.shape) < h_prob).astype(np.float64)

            # Negative phase + mask enforcement
            v_recon_prob = sigmoid(h_data @ W.T)
            v_recon_prob = v_recon_prob * m_batch
            v_recon = (rng.random(v_recon_prob.shape) < v_recon_prob).astype(np.float64)
            v_recon = v_recon * m_batch

            h_recon_prob = sigmoid(v_recon @ W + b_h)
            h_recon = (rng.random(h_recon_prob.shape) < h_recon_prob).astype(np.float64)

            pos = v_data.T @ h_data
            neg = v_recon.T @ h_recon
            dW = LR * (pos - neg) / bs
            epoch_delta += dW

            db_h = LR * (h_prob.mean(axis=0) - h_recon_prob.mean(axis=0))
            b_h += db_h

        W += epoch_delta
        dt = time.perf_counter() - t0

        ep = epoch + 1
        if ep in (1, 5, EPOCHS):
            print(f"  … computing masked MSE (chunked) …", flush=True)
            mse = reconstruction_mse(X, M, W, b_h)
            mse_log[ep] = mse
            print(f"  Epoch {ep:02d} | masked recon MSE: {mse:.6f} | epoch wall {dt:.1f}s", flush=True)
        else:
            print(f"  Epoch {ep:02d} | epoch wall {dt:.1f}s", flush=True)

        if ep == 1:
            print(
                f"  ⏱ epoch 1 wall time: {dt:.1f}s  → est. full {EPOCHS} epochs ≈ {dt * EPOCHS / 60:.1f} min "
                f"(one channel; ×2 for A1+A2)",
                flush=True,
            )

    elapsed = time.perf_counter() - t0_all
    print(f"  Done in {elapsed / 60:.1f} min", flush=True)
    return W, b_h, mse_log, elapsed


W1, bh1, mse1, t1 = train_rbm(
    X1, M, N_HIDDEN, seed=INIT_SEED, channel_name="RBM_A1 (user rating)"
)
W2, bh2, mse2, t2 = train_rbm(
    X2, M, N_HIDDEN, seed=INIT_SEED, channel_name="RBM_A2 (mirror)"
)
total_train_sec = t1 + t2
print(f"\nTotal training wall time: {total_train_sec / 60:.1f} min", flush=True)



=== Training RBM_A1 (user rating) ===
  samples=2000, visible=131290, hidden=128
  lr=0.01, batch=500, epochs=10, init_seed=42
  … epoch 1/10 training
  … computing masked MSE (chunked) …
  Epoch 01 | masked recon MSE: 0.244173 | epoch wall 15.2s
  ⏱ epoch 1 wall time: 15.2s  → est. full 10 epochs ≈ 2.5 min (one channel; ×2 for A1+A2)
  … epoch 2/10 training
  Epoch 02 | epoch wall 13.9s
  … epoch 3/10 training
  Epoch 03 | epoch wall 13.3s
  … epoch 4/10 training
  Epoch 04 | epoch wall 12.8s
  … epoch 5/10 training
  … computing masked MSE (chunked) …
  Epoch 05 | masked recon MSE: 0.234025 | epoch wall 14.3s
  … epoch 6/10 training
  Epoch 06 | epoch wall 13.1s
  … epoch 7/10 training
  Epoch 07 | epoch wall 13.3s
  … epoch 8/10 training
  Epoch 08 | epoch wall 14.2s
  … epoch 9/10 training
  Epoch 09 | epoch wall 13.0s
  … epoch 10/10 training
  … computing masked MSE (chunked) …
  Epoch 10 | masked recon MSE: 0.205951 | epoch wall 13.6s
  Done in 2.7 min

=== Training RBM_A2 (mir

## Part 3 — Save artifacts

In [4]:
paths = {
    "W1": proc / "rbmA1_weights_5k.npy",
    "W2": proc / "rbmA2_weights_5k.npy",
    "bh1": proc / "rbmA1_bias_hidden_5k.npy",
    "bh2": proc / "rbmA2_bias_hidden_5k.npy",
}

np.save(paths["W1"], W1)
np.save(paths["W2"], W2)
np.save(paths["bh1"], bh1)
np.save(paths["bh2"], bh2)

meta = {
    "EPOCHS": EPOCHS,
    "N_HIDDEN": N_HIDDEN,
    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "INIT_SEED": INIT_SEED,
    "encoding": "Rating Mirror",
    "N_USERS_MIRROR": N_USERS_MIRROR,
    "deltaW_logging": False,
}
np.save(proc / "rbmA_5k_train_meta.npy", meta)

for k, p in paths.items():
    print(f"Saved {k}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")
print(f"Saved meta: {proc / 'rbmA_5k_train_meta.npy'}")


Saved W1: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmA1_weights_5k.npy  (134.4 MB)
Saved W2: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmA2_weights_5k.npy  (134.4 MB)
Saved bh1: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmA1_bias_hidden_5k.npy  (0.0 MB)
Saved bh2: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmA2_bias_hidden_5k.npy  (0.0 MB)
Saved meta: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbmA_5k_train_meta.npy


## Part 3.5 — Mixed-sign occupancy

Pre-sigmoid dual-channel hidden fields:
\(z_1 = X_1 W_1 + b_{h1}\), \(z_2 = X_2 W_2 + b_{h2}\).

A hidden unit is **mixed-sign** for a user when \(\mathrm{sign}(z_1) \neq \mathrm{sign}(z_2)\).
Report the fraction of mixed-sign \((\mathrm{user}, \mathrm{hidden})\) pairs.


In [5]:
z1 = X1 @ W1 + bh1
z2 = X2 @ W2 + bh2
n_hidden = W1.shape[1]
assert z1.shape == z2.shape == (X1.shape[0], n_hidden)

mixed = np.sign(z1) != np.sign(z2)
mixed_sign_rate = float(mixed.mean())
print(f"Mixed-sign occupancy (Rating Mirror): {100.0 * mixed_sign_rate:.4f}%")


Mixed-sign occupancy (Rating Mirror): 20.2547%


## Part 4 — Verification

In [6]:
def print_W_stats(name, W):
    print(f"{name}: shape={W.shape}")
    print(
        f"  mean={W.mean():.6f}  std={W.std():.6f}  "
        f"min={W.min():.6f}  max={W.max():.6f}"
    )


print("=== Final weights ===")
print_W_stats("W1 (RBM_A1)", W1)
print_W_stats("W2 (RBM_A2)", W2)

print("\n=== Masked recon MSE ===")
for ep in (1, 5, EPOCHS):
    print(
        f"  epoch {ep:2d}:  A1={mse1.get(ep, float('nan')):.6f}  "
        f"A2={mse2.get(ep, float('nan')):.6f}"
    )

# Direct analog of the 51-user experiment: flattened Pearson r(W1, W2)
r_w = float(np.corrcoef(W1.ravel(), W2.ravel())[0, 1])
print(f"\nPearson r(W1.ravel(), W2.ravel()) = {r_w:.6f}")
print("(same-shape Rating Mirror channels; compare to original 51-user r ≈ 0.908)")

print(f"\nTotal training time: {total_train_sec / 60:.1f} min ({total_train_sec:.0f}s)")
print("\nAll checks passed.")

=== Final weights ===
W1 (RBM_A1): shape=(131290, 128)
  mean=-0.000716  std=0.010342  min=-0.061305  max=0.087936
W2 (RBM_A2): shape=(131290, 128)
  mean=-0.000718  std=0.010334  min=-0.062945  max=0.094309

=== Masked recon MSE ===
  epoch  1:  A1=0.244173  A2=0.244235
  epoch  5:  A1=0.234025  A2=0.234169
  epoch 10:  A1=0.205951  A2=0.204908

Pearson r(W1.ravel(), W2.ravel()) = 0.927089
(same-shape Rating Mirror channels; compare to original 51-user r ≈ 0.908)

Total training time: 5.4 min (327s)

All checks passed.
